# Music Annotation Project — Notebook 32
## V8.2c Whole-Population Label-Sensitivity and Thin-Structure Audit

**Runtime:** CPU is sufficient.

This is a development-only population audit. It does **not** retrain, refit, recalibrate, change thresholds, or generate a replacement corrected page.

It replays the exact V8.2 clean-truth construction for every eligible OOF pixel, measures Euclidean and Chebyshev distance to clean printed ink, identifies current ANNOTATION labels exactly two pixels from clean ink, and creates an explicitly labelled **morphological proxy** for thin continuous clean structures. The proxy is not a semantic recogniser for staff lines, hairpins, stems or slurs; those categories remain for human review.

### Primary hypotheses

1. A material subset of current `ANNOTATION` truth lies exactly two Chebyshev pixels from clean ink.
2. Thin or slightly displaced printed structures are disproportionately assigned `UNCERTAIN` or `ANNOTATION` by the current erosion-heavy truth rule.
3. These effects vary by page and source group rather than being isolated to one click-inspected page.
4. The V8.2 classifier probabilities follow the supplied labels; this notebook audits the labels and geometry rather than retuning the classifier.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib scipy scikit-image

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, shutil, sys, platform

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import distance_transform_edt
from skimage.morphology import skeletonize

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
V82_ROOT = ROOT / 'observed_print_classifier_v8_2'
OUTPUT_ROOT = ROOT / 'observed_print_classifier_v8_2c_label_sensitivity'
SOURCE_V82_RUN_OVERRIDE = None

# Exact V8.2 truth construction constants.
ADAPTIVE_BLOCK_SIZE = 41
ADAPTIVE_C = 15
TRUTH_KERNEL_SIZE = 3
PATCH = 33
HALF = PATCH // 2

# Exploratory, auditable morphology-proxy controls.
# These DO NOT relabel data automatically and are not musical-symbol semantics.
THIN_MAX_LOCAL_WIDTH_PX = 4.0
MIN_LOCAL_SKELETON_PIXELS = 8
MIN_LOCAL_SKELETON_COMPONENT = 6
MIN_LOCAL_LINEARITY = 0.60

DISTANCE2_REVIEW_LIMIT = 500
REVIEW_CROP_LIMIT = 120
CROP_RADIUS = 40
CONTACT_SHEET_ROWS = 10
COPY_SOURCE_RECORD = True

RUN = datetime.now().strftime('%Y%m%d_%H%M%S') + '_v8_2c_label_sensitivity_audit'
OUT = OUTPUT_ROOT / RUN
RECORD = OUT / 'source_v82_record'
TABLES = OUT / 'tables'
AUDIT = OUT / 'audit'
VISUAL = OUT / 'visual_audit'
CROPS = VISUAL / 'distance2_review_crops'
OVERLAYS = VISUAL / 'page_overlays'
for p in [OUT, RECORD, TABLES, AUDIT, VISUAL, CROPS, OVERLAYS]:
    p.mkdir(parents=True, exist_ok=False)

TRUTH_NAMES = {0:'ANNOTATION', 1:'PRINT', 2:'UNCERTAIN'}
print('Output:', OUT)

## 2. Integrity helpers and completed V8.2 source resolution

In [ ]:
SUPPORTED = {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()

def image_map(folder):
    folder = Path(folder)
    return {p.stem:p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if folder.exists() else {}

REQUIRED_SOURCE = [
    'experiment_identity.json',
    'tables/all_eligible_oof_predictions.parquet',
    'tables/eligible_pixel_manifest.parquet',
    'audit/input_and_label_audit.csv',
    'inputs/channel_schema.json',
    'inputs/fold_map.csv',
]

def complete_runs(root):
    hits=[]
    if Path(root).exists():
        for p in Path(root).iterdir():
            if p.is_dir() and all((p/r).exists() for r in REQUIRED_SOURCE):
                hits.append(p)
    return sorted(hits, key=lambda p:p.stat().st_mtime)

if SOURCE_V82_RUN_OVERRIDE is not None:
    SOURCE = Path(SOURCE_V82_RUN_OVERRIDE)
    missing=[r for r in REQUIRED_SOURCE if not (SOURCE/r).exists()]
    if missing:
        raise RuntimeError(f'Incomplete V8.2 source override: {missing}')
else:
    hits=complete_runs(V82_ROOT)
    if not hits:
        raise RuntimeError(f'No complete V8.2 run found below {V82_ROOT}')
    SOURCE=hits[-1]

IDENTITY=json.loads((SOURCE/'experiment_identity.json').read_text())
if IDENTITY.get('version')!='8.2-development-whole-page':
    raise RuntimeError(f"Unexpected source identity: {IDENTITY.get('version')}")
if IDENTITY.get('independent_data_used') is not False:
    raise RuntimeError('Notebook 32 may use only the development-only V8.2 OOF run.')

PRED_PATH=SOURCE/'tables'/'all_eligible_oof_predictions.parquet'
ELIGIBLE_PATH=SOURCE/'tables'/'eligible_pixel_manifest.parquet'
INPUT_AUDIT_PATH=SOURCE/'audit'/'input_and_label_audit.csv'
full=pd.read_parquet(PRED_PATH)
eligible=pd.read_parquet(ELIGIBLE_PATH)
input_audit=pd.read_csv(INPUT_AUDIT_PATH)

required={'page','source_group','y','x','truth','fold','raw_print','p_print'}
missing=required-set(full.columns)
if missing:
    raise RuntimeError(f'Prediction table missing columns: {sorted(missing)}')
for c in ['y','x','truth','fold']:
    full[c]=pd.to_numeric(full[c],errors='raise').astype(int)
for c in ['raw_print','p_print']:
    full[c]=pd.to_numeric(full[c],errors='raise').astype(np.float64)
full['page']=full.page.astype(str)
full['source_group']=full.source_group.astype(str)
if full.duplicated(['page','y','x']).any():
    raise RuntimeError('Duplicate page/y/x rows in V8.2 predictions.')
if not np.isfinite(full[['raw_print','p_print']].to_numpy()).all():
    raise RuntimeError('Non-finite prediction values found.')

key=['page','source_group','y','x','truth','fold']
eligible=eligible.copy()
eligible['page']=eligible.page.astype(str)
eligible['source_group']=eligible.source_group.astype(str)
for c in ['y','x','truth','fold']:
    eligible[c]=pd.to_numeric(eligible[c],errors='raise').astype(int)
if len(full)!=len(eligible) or len(full.merge(eligible[key],on=key,how='inner'))!=len(full):
    raise RuntimeError('OOF predictions and eligible-pixel manifest do not match exactly.')

print('Source:', SOURCE)
print('Rows:',len(full),'Pages:',full.page.nunique(),'Source groups:',full.source_group.nunique())

## 3. Snapshot the exact source evidence used by this audit

In [ ]:
source_files=[
    SOURCE/'experiment_identity.json',
    SOURCE/'development_summary.json',
    SOURCE/'executive_summary.txt',
    SOURCE/'tables'/'all_eligible_oof_predictions.parquet',
    SOURCE/'tables'/'eligible_pixel_manifest.parquet',
    SOURCE/'audit'/'input_and_label_audit.csv',
    SOURCE/'inputs'/'channel_schema.json',
    SOURCE/'inputs'/'fold_map.csv',
    SOURCE/'inputs'/'development_score_manifest.csv',
]
copy_rows=[]
if COPY_SOURCE_RECORD:
    for src in source_files:
        if not src.exists():
            continue
        rel=src.relative_to(SOURCE)
        dst=RECORD/rel
        dst.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(src,dst)
        if sha256_file(src)!=sha256_file(dst):
            raise RuntimeError(f'Copied source hash mismatch: {rel}')
        copy_rows.append({'relative_path':str(rel),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})

pd.DataFrame(copy_rows).to_csv(AUDIT/'copied_source_record_inventory.csv',index=False)
source_record={
    'source_run':str(SOURCE),
    'source_identity_sha256':sha256_file(SOURCE/'experiment_identity.json'),
    'predictions_sha256':sha256_file(PRED_PATH),
    'eligible_manifest_sha256':sha256_file(ELIGIBLE_PATH),
    'input_audit_sha256':sha256_file(INPUT_AUDIT_PATH),
    'copied_files':len(copy_rows),
}
(OUT/'source_v82_record.json').write_text(json.dumps(source_record,indent=2))
print(json.dumps(source_record,indent=2))

## 4. Resolve clean pages and exact V7.5 contextual arrays

In [ ]:
CLEAN_MAP=image_map(ROOT/'source'/'clean')
OBSERVED_MAP=image_map(ROOT/'aligned')
V75=Path(IDENTITY['v7_5_reference'])
status=pd.read_csv(V75/'tables'/'cache_status.csv')
CACHE={str(r.page):Path(r.cache) for _,r in status.iterrows()}

pages=sorted(full.page.unique())
audit_by_page=input_audit.copy()
audit_by_page['page']=audit_by_page.page.astype(str)
audit_by_page=audit_by_page.set_index('page')

for page in pages:
    if page not in CLEAN_MAP: raise RuntimeError(f'{page}: clean page not found')
    if page not in OBSERVED_MAP: raise RuntimeError(f'{page}: aligned observed page not found')
    if page not in CACHE: raise RuntimeError(f'{page}: V7.5 cache not found')
    if page not in audit_by_page.index: raise RuntimeError(f'{page}: absent from V8.2 input audit')
    expected=str(audit_by_page.loc[page,'clean_sha256'])
    actual=sha256_file(CLEAN_MAP[page])
    if expected!=actual:
        raise RuntimeError(f'{page}: clean-source hash mismatch against V8.2 input audit')
    expected_obs=str(audit_by_page.loc[page,'observed_sha256'])
    actual_obs=sha256_file(OBSERVED_MAP[page])
    if expected_obs!=actual_obs:
        raise RuntimeError(f'{page}: observed-source hash mismatch against V8.2 input audit')

pd.DataFrame([
    {'page':p,'clean_path':str(CLEAN_MAP[p]),'clean_sha256':sha256_file(CLEAN_MAP[p]),
     'observed_path':str(OBSERVED_MAP[p]),'observed_sha256':sha256_file(OBSERVED_MAP[p]),
     'cache_path':str(CACHE[p]),'cache_sha256':sha256_file(CACHE[p])}
    for p in pages
]).to_csv(AUDIT/'resolved_input_files.csv',index=False)
print('Hash-verified clean/observed/cache inputs for',len(pages),'pages')

## 5. Exact truth replay and distance/morphology helpers

The current V8.2 truth is replayed exactly:

- `PRINT`: eligible and `near` and not `boundary` (which is effectively eroded clean-ink interior);
- `ANNOTATION`: eligible and not `near` and not `boundary`;
- `UNCERTAIN`: eligible and `boundary`.

The thin-continuous flag below is an explicitly parameterised morphology proxy. It does not claim to identify staff lines, hairpins, stems or slurs.

In [ ]:
def printed_ink(clean):
    return cv2.adaptiveThreshold(
        clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,ADAPTIVE_BLOCK_SIZE,ADAPTIVE_C
    )>0

def truth_maps(clean):
    ink=printed_ink(clean)
    k=np.ones((TRUTH_KERNEL_SIZE,TRUTH_KERNEL_SIZE),np.uint8)
    near=cv2.dilate(ink.astype(np.uint8),k)>0
    eroded=cv2.erode(ink.astype(np.uint8),k)>0
    boundary=np.logical_xor(near,eroded)
    return ink,near,eroded,boundary

def truth_from_maps(near,boundary,ys,xs):
    out=np.full(len(ys),-1,np.int8)
    b=boundary[ys,xs]
    n=near[ys,xs]
    out[n & ~b]=1
    out[~n & ~b]=0
    out[b]=2
    return out

def route_name(preserve,remove,ambiguous,ys,xs):
    out=np.full(len(ys),'none',object)
    out[preserve[ys,xs]]='preserve'
    out[remove[ys,xs]]='remove'
    out[ambiguous[ys,xs]]='ambiguous'
    return out

def local_skeleton_features(skeleton,y,x,radius=HALF):
    h,w=skeleton.shape
    y0=max(0,y-radius); y1=min(h,y+radius+1)
    x0=max(0,x-radius); x1=min(w,x+radius+1)
    patch=skeleton[y0:y1,x0:x1].astype(np.uint8)
    count=int(patch.sum())
    if count:
        n,lab,stats,_=cv2.connectedComponentsWithStats(patch,8)
        largest=int(stats[1:,cv2.CC_STAT_AREA].max()) if n>1 else 0
        yy,xx=np.where(patch>0)
        if len(xx)>=3:
            pts=np.column_stack([xx,yy]).astype(np.float64)
            cov=np.cov(pts,rowvar=False)
            vals=np.linalg.eigvalsh(cov)
            vals=np.sort(np.maximum(vals,0))[::-1]
            linearity=float(1.0-vals[1]/max(vals[0],1e-12))
            angle=float(np.degrees(np.arctan2(*np.linalg.eigh(cov)[1][:,np.argmax(np.linalg.eigh(cov)[0])][::-1])))
        else:
            linearity=0.0; angle=np.nan
    else:
        largest=0; linearity=0.0; angle=np.nan
    return count,largest,linearity,angle

def distance_bin(d):
    d=int(round(float(d)))
    return str(d) if d<=3 else '4+'

## 6. Whole-population enrichment

In [ ]:
enriched_pages=[]
page_summary=[]

for page,g0 in full.groupby('page',sort=False):
    print('Processing',page,'rows',len(g0))
    g=g0.copy().reset_index(drop=True)
    clean=cv2.imread(str(CLEAN_MAP[page]),cv2.IMREAD_GRAYSCALE)
    observed=cv2.imread(str(OBSERVED_MAP[page]),cv2.IMREAD_GRAYSCALE)
    if clean is None or observed is None or clean.shape!=observed.shape:
        raise RuntimeError(f'{page}: clean/observed read or shape failure')

    with np.load(CACHE[page]) as z:
        required=['mask','g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact']
        miss=[k for k in required if k not in z.files]
        if miss: raise RuntimeError(f'{page}: cache missing {miss}')
        arr={k:z[k]>0 for k in required}
    if any(v.shape!=clean.shape for v in arr.values()):
        raise RuntimeError(f'{page}: cache shape mismatch')

    ink,near,eroded,boundary=truth_maps(clean)
    nonink=(~ink).astype(np.uint8)
    cheb=cv2.distanceTransform(nonink,cv2.DIST_C,3).astype(np.float32)
    euclid,nearest_indices=distance_transform_edt(~ink,return_indices=True)
    ink_radius=cv2.distanceTransform(ink.astype(np.uint8),cv2.DIST_L2,5).astype(np.float32)
    skeleton=skeletonize(ink)

    ys=g.y.to_numpy(int); xs=g.x.to_numpy(int)
    h,w=clean.shape
    if (ys<0).any() or (xs<0).any() or (ys>=h).any() or (xs>=w).any():
        raise RuntimeError(f'{page}: prediction coordinate outside page')

    rederived=truth_from_maps(near,boundary,ys,xs)
    mismatch=rederived!=g.truth.to_numpy(int)
    if mismatch.any():
        ex=g.loc[mismatch,['page','x','y','truth']].head(20).to_dict('records')
        raise RuntimeError(f'{page}: stored truth differs from exact replay; examples={ex}')

    ny=nearest_indices[0,ys,xs].astype(int)
    nx=nearest_indices[1,ys,xs].astype(int)
    dcheb=np.rint(cheb[ys,xs]).astype(int)
    deuclid=euclid[ys,xs].astype(np.float32)
    nearest_width=2.0*ink_radius[ny,nx]

    # Fast 33×33 context counts.
    ink_count=cv2.boxFilter(ink.astype(np.float32),-1,(PATCH,PATCH),normalize=False,borderType=cv2.BORDER_REFLECT)[ys,xs]
    skel_count=cv2.boxFilter(skeleton.astype(np.float32),-1,(PATCH,PATCH),normalize=False,borderType=cv2.BORDER_REFLECT)[ys,xs]

    g['truth_name']=g.truth.map(TRUTH_NAMES)
    g['clean_pixel_value']=clean[ys,xs]
    g['clean_ink_at_centre']=ink[ys,xs]
    g['clean_near_at_centre']=near[ys,xs]
    g['clean_eroded_at_centre']=eroded[ys,xs]
    g['clean_boundary_at_centre']=boundary[ys,xs]
    g['rederived_truth']=rederived
    g['nearest_clean_ink_x']=nx
    g['nearest_clean_ink_y']=ny
    g['nearest_clean_ink_chebyshev']=dcheb
    g['nearest_clean_ink_euclidean']=deuclid
    g['distance_bin']=pd.Categorical([distance_bin(v) for v in dcheb],categories=['0','1','2','3','4+'],ordered=True)
    g['nearest_clean_ink_local_width']=nearest_width
    g['clean_ink_pixels_33']=ink_count.astype(int)
    g['clean_skeleton_pixels_33']=skel_count.astype(int)

    g['route']=route_name(arr['g80_preserve'],arr['g80_remove'],arr['g80_ambiguous'],ys,xs)
    g['horizontal_support_centre']=arr['g80_horizontal'][ys,xs]
    g['vertical_support_centre']=arr['g80_vertical'][ys,xs]
    g['compact_support_centre']=arr['g80_compact'][ys,xs]

    # Detailed morphology only for the critical current-ANNOTATION distance-two population.
    critical=(g.truth.to_numpy(int)==0)&(dcheb==2)
    local_largest=np.zeros(len(g),np.int32)
    local_linearity=np.full(len(g),np.nan,np.float32)
    local_angle=np.full(len(g),np.nan,np.float32)
    for j in np.where(critical)[0]:
        cnt,largest,linearity,angle=local_skeleton_features(skeleton,int(ys[j]),int(xs[j]))
        local_largest[j]=largest
        local_linearity[j]=linearity
        local_angle[j]=angle
    g['local_skeleton_largest_component_33']=local_largest
    g['local_skeleton_linearity_33']=local_linearity
    g['local_skeleton_orientation_deg_33']=local_angle
    g['thin_continuous_proxy']=(
        critical &
        (nearest_width<=THIN_MAX_LOCAL_WIDTH_PX) &
        (skel_count>=MIN_LOCAL_SKELETON_PIXELS) &
        (local_largest>=MIN_LOCAL_SKELETON_COMPONENT) &
        (np.nan_to_num(local_linearity,nan=0.0)>=MIN_LOCAL_LINEARITY)
    )

    # Explicit sensitivity labels; no model is retrained with these labels here.
    g['variant_tolerance1_truth']=np.where(dcheb<=1,1,0).astype(np.int8)
    g['variant_tolerance2_truth']=np.where(dcheb<=2,1,0).astype(np.int8)
    g['variant_distance2_to_uncertain_truth']=g.truth.to_numpy(np.int8)
    g.loc[(g.truth==0)&(g.nearest_clean_ink_chebyshev==2),'variant_distance2_to_uncertain_truth']=2
    g['variant_thin_proxy_to_uncertain_truth']=g.truth.to_numpy(np.int8)
    g.loc[g.thin_continuous_proxy,'variant_thin_proxy_to_uncertain_truth']=2

    enriched_pages.append(g)
    ann=(g.truth==0)
    ann2=ann&(g.nearest_clean_ink_chebyshev==2)
    page_summary.append({
        'page':page,'source_group':str(g.source_group.iloc[0]),
        'eligible_pixels':len(g),'annotation_pixels':int(ann.sum()),
        'annotation_distance2_pixels':int(ann2.sum()),
        'annotation_distance2_fraction':float(ann2.sum()/max(ann.sum(),1)),
        'distance2_thin_continuous_proxy_pixels':int(g.thin_continuous_proxy.sum()),
    })

enriched=pd.concat(enriched_pages,ignore_index=True)
enriched.to_parquet(TABLES/'all_eligible_label_sensitivity.parquet',index=False)
pd.DataFrame(page_summary).to_csv(TABLES/'distance2_annotation_by_page.csv',index=False)
print('Enriched rows:',len(enriched))

## 7. Exact answer: target fraction, page/source distribution and truth-distance histograms

In [ ]:
annotation=enriched[enriched.truth==0]
annotation_d2=annotation[annotation.nearest_clean_ink_chebyshev==2].copy()
thin_d2=annotation_d2[annotation_d2.thin_continuous_proxy].copy()

answer={
    'eligible_annotation_pixels':int(len(annotation)),
    'annotation_exactly_two_chebyshev_pixels_from_clean_ink':int(len(annotation_d2)),
    'fraction_annotation_exactly_two_chebyshev_pixels_from_clean_ink':float(len(annotation_d2)/max(len(annotation),1)),
    'distance2_annotation_thin_continuous_proxy_pixels':int(len(thin_d2)),
    'fraction_distance2_annotation_meeting_thin_continuous_proxy':float(len(thin_d2)/max(len(annotation_d2),1)),
    'proxy_definition':{
        'nearest_clean_ink_local_width_max_px':THIN_MAX_LOCAL_WIDTH_PX,
        'minimum_clean_skeleton_pixels_in_33x33':MIN_LOCAL_SKELETON_PIXELS,
        'minimum_largest_skeleton_component_in_33x33':MIN_LOCAL_SKELETON_COMPONENT,
        'minimum_skeleton_linearity_in_33x33':MIN_LOCAL_LINEARITY,
        'semantic_symbol_identity_claimed':False,
    },
}
(OUT/'primary_answer.json').write_text(json.dumps(answer,indent=2))
print(json.dumps(answer,indent=2))

hist=(enriched.groupby(['truth_name','distance_bin'],observed=False,as_index=False).size().rename(columns={'size':'pixels'}))
totals=enriched.groupby('truth_name',as_index=False).size().rename(columns={'size':'truth_pixels'})
hist=hist.merge(totals,on='truth_name')
hist['fraction_within_truth']=hist.pixels/hist.truth_pixels
hist.to_csv(TABLES/'truth_by_nearest_clean_ink_distance.csv',index=False)

ann_group=(annotation.groupby(['source_group','distance_bin'],observed=False,as_index=False).size().rename(columns={'size':'pixels'}))
ann_group.to_csv(TABLES/'annotation_distance_by_source_group.csv',index=False)

pd.DataFrame(page_summary).groupby('source_group',as_index=False).agg(
    pages=('page','nunique'),annotation_pixels=('annotation_pixels','sum'),
    annotation_distance2_pixels=('annotation_distance2_pixels','sum'),
    distance2_thin_continuous_proxy_pixels=('distance2_thin_continuous_proxy_pixels','sum')
).assign(annotation_distance2_fraction=lambda d:d.annotation_distance2_pixels/np.maximum(d.annotation_pixels,1)).to_csv(TABLES/'distance2_annotation_by_source_group.csv',index=False)

annotation_d2.to_parquet(TABLES/'annotation_distance2_candidates.parquet',index=False)
annotation_d2.to_csv(TABLES/'annotation_distance2_candidates.csv',index=False)
thin_d2.to_csv(TABLES/'annotation_distance2_thin_continuous_proxy.csv',index=False)

## 8. Alternative-label transition audit and probability behaviour

In [ ]:
variant_cols=[
    'variant_tolerance1_truth',
    'variant_tolerance2_truth',
    'variant_distance2_to_uncertain_truth',
    'variant_thin_proxy_to_uncertain_truth',
]
transition_rows=[]
for col in variant_cols:
    q=(enriched.groupby(['truth',col],as_index=False).size().rename(columns={'size':'pixels',col:'variant_truth'}))
    q['variant']=col
    q['current_truth_name']=q.truth.map(TRUTH_NAMES)
    q['variant_truth_name']=q.variant_truth.map(TRUTH_NAMES)
    transition_rows.append(q)
transitions=pd.concat(transition_rows,ignore_index=True)
transitions.to_csv(TABLES/'alternative_truth_transition_summary.csv',index=False)

by_page=[]
for col in variant_cols:
    q=enriched[enriched.truth!=enriched[col]].groupby(['page','source_group','truth',col],as_index=False).size().rename(columns={'size':'pixels',col:'variant_truth'})
    q['variant']=col
    by_page.append(q)
pd.concat(by_page,ignore_index=True).to_csv(TABLES/'alternative_truth_transitions_by_page.csv',index=False)

prob=(enriched.groupby(['truth_name','distance_bin'],observed=False).p_print.agg(
    pixels='size',mean='mean',median='median',minimum='min',maximum='max'
).reset_index())
quant=enriched.groupby(['truth_name','distance_bin'],observed=False).p_print.quantile([.90,.95,.99]).unstack().reset_index().rename(columns={.9:'p90',.95:'p95',.99:'p99'})
prob=prob.merge(quant,on=['truth_name','distance_bin'],how='left')
prob.to_csv(TABLES/'p_print_by_truth_and_clean_ink_distance.csv',index=False)

route=(enriched.groupby(['truth_name','distance_bin','route'],observed=True,as_index=False).agg(
    pixels=('p_print','size'),mean_p_print=('p_print','mean'),
    horizontal_support_rate=('horizontal_support_centre','mean'),
    vertical_support_rate=('vertical_support_centre','mean'),
    compact_support_rate=('compact_support_centre','mean'),
))
route.to_csv(TABLES/'route_support_by_truth_distance.csv',index=False)

## 9. Plots and page-level overlays

In [ ]:
# Distance distribution by truth.
pivot=hist.pivot(index='distance_bin',columns='truth_name',values='fraction_within_truth').fillna(0).reindex(['0','1','2','3','4+'])
ax=pivot.plot(kind='bar',figsize=(11,6))
ax.set_title('Nearest clean-ink Chebyshev distance by current V8.2 truth')
ax.set_xlabel('Distance (pixels)'); ax.set_ylabel('Fraction within truth class')
plt.xticks(rotation=0); plt.tight_layout(); plt.savefig(VISUAL/'truth_distance_distribution.png',dpi=160); plt.close()

# Annotation distance-two fractions by page.
ps=pd.DataFrame(page_summary).sort_values('annotation_distance2_fraction',ascending=False)
fig,ax=plt.subplots(figsize=(14,6))
ax.bar(ps.page,ps.annotation_distance2_fraction)
ax.set_title('Current ANNOTATION pixels exactly two pixels from clean ink, by page')
ax.set_ylabel('Fraction of ANNOTATION truth'); ax.tick_params(axis='x',rotation=90)
plt.tight_layout(); plt.savefig(VISUAL/'annotation_distance2_fraction_by_page.png',dpi=160); plt.close()

# Probability by truth/distance.
fig,ax=plt.subplots(figsize=(11,6))
for truth,q in prob.groupby('truth_name'):
    q=q.set_index('distance_bin').reindex(['0','1','2','3','4+'])
    ax.plot(['0','1','2','3','4+'],q['median'],marker='o',label=truth)
ax.set_title('Median calibrated PRINT probability by truth and clean-ink distance')
ax.set_xlabel('Chebyshev distance'); ax.set_ylabel('Median p_print'); ax.legend()
plt.tight_layout(); plt.savefig(VISUAL/'p_print_median_by_truth_distance.png',dpi=160); plt.close()

# Page overlays: red = all distance-two ANNOTATION; lime = thin-continuous proxy subset.
for page,g in annotation_d2.groupby('page'):
    obs=cv2.imread(str(OBSERVED_MAP[page]),cv2.IMREAD_GRAYSCALE)
    base=cv2.cvtColor(obs,cv2.COLOR_GRAY2BGR)
    all_mask=np.zeros(obs.shape,bool); all_mask[g.y.to_numpy(int),g.x.to_numpy(int)]=True
    proxy=g[g.thin_continuous_proxy]
    proxy_mask=np.zeros(obs.shape,bool)
    if len(proxy): proxy_mask[proxy.y.to_numpy(int),proxy.x.to_numpy(int)]=True
    colour=base.copy(); colour[all_mask]=(0,0,255); colour[proxy_mask]=(80,255,80)
    active=all_mask|proxy_mask
    out=base.copy(); out[active]=cv2.addWeighted(base,.35,colour,.65,0)[active]
    cv2.putText(out,'RED: ANNOTATION distance=2  LIME: thin-continuous proxy',(12,28),cv2.FONT_HERSHEY_SIMPLEX,.65,(20,20,20),2,cv2.LINE_AA)
    cv2.imwrite(str(OVERLAYS/f'{page}_annotation_distance2_overlay.png'),out)

## 10. Ranked review queue and contact sheets

The queue deliberately avoids claiming symbol semantics. Human review may subsequently assign categories such as staff line, hairpin, stem, slur, other printed structure, not printed structure, or uncertain.

In [ ]:
review=annotation_d2.copy()
review['proxy_rank_score']=(
    review.thin_continuous_proxy.astype(int)*1000000 +
    review.local_skeleton_largest_component_33.fillna(0)*1000 +
    review.clean_skeleton_pixels_33.fillna(0)*10 +
    review.local_skeleton_linearity_33.fillna(0) -
    review.p_print
)
review=review.sort_values(['thin_continuous_proxy','proxy_rank_score'],ascending=[False,False]).head(DISTANCE2_REVIEW_LIMIT).copy()
review['human_symbol_category']=''
review['human_ownership']=''
review['human_registration_comment']=''
review['review_status']='unreviewed'
review.to_csv(TABLES/'distance2_annotation_review_queue.csv',index=False)

TRUTH_BGR={0:(0,0,220),1:(0,200,0),2:(0,190,255)}

def crop_with_pad(img,y,x,radius=CROP_RADIUS,pad_value=255):
    h,w=img.shape[:2]
    y0,y1=y-radius,y+radius+1; x0,x1=x-radius,x+radius+1
    top=max(0,-y0); left=max(0,-x0); bottom=max(0,y1-h); right=max(0,x1-w)
    y0=max(0,y0); x0=max(0,x0); y1=min(h,y1); x1=min(w,x1)
    crop=img[y0:y1,x0:x1]
    return cv2.copyMakeBorder(crop,top,bottom,left,right,cv2.BORDER_CONSTANT,value=pad_value)

def registration_overlay(obs_crop,ink_crop,dx,dy):
    out=cv2.cvtColor(obs_crop,cv2.COLOR_GRAY2BGR)
    contour=cv2.morphologyEx(ink_crop.astype(np.uint8),cv2.MORPH_GRADIENT,np.ones((3,3),np.uint8))>0
    out[contour]=(0,255,0)
    c=(CROP_RADIUS,CROP_RADIUS)
    n=(CROP_RADIUS+int(dx),CROP_RADIUS+int(dy))
    cv2.drawMarker(out,c,(255,0,255),cv2.MARKER_CROSS,15,1)
    if 0<=n[0]<out.shape[1] and 0<=n[1]<out.shape[0]:
        cv2.circle(out,n,4,(255,255,0),1); cv2.line(out,c,n,(255,255,0),1)
    return out

def truth_colour_patch(near,boundary,y,x):
    label=np.full(near.shape,0,np.uint8)
    label[near&~boundary]=1; label[boundary]=2
    patch=crop_with_pad(label,y,x,pad_value=0)
    out=np.full((*patch.shape,3),235,np.uint8)
    for k,bgr in TRUTH_BGR.items(): out[patch==k]=bgr
    cv2.drawMarker(out,(CROP_RADIUS,CROP_RADIUS),(255,0,255),cv2.MARKER_CROSS,15,1)
    return out

crop_records=[]
for _,r in review.head(REVIEW_CROP_LIMIT).iterrows():
    page=str(r.page); x=int(r.x); y=int(r.y)
    clean=cv2.imread(str(CLEAN_MAP[page]),0); obs=cv2.imread(str(OBSERVED_MAP[page]),0)
    ink,near,eroded,boundary=truth_maps(clean)
    obs_crop=crop_with_pad(obs,y,x); clean_crop=crop_with_pad(clean,y,x)
    ink_crop=crop_with_pad(ink.astype(np.uint8),y,x,pad_value=0)>0
    dx=int(r.nearest_clean_ink_x)-x; dy=int(r.nearest_clean_ink_y)-y
    overlay=registration_overlay(obs_crop,ink_crop,dx,dy)
    truth_patch=truth_colour_patch(near,boundary,y,x)
    panels=[cv2.cvtColor(obs_crop,cv2.COLOR_GRAY2BGR),cv2.cvtColor(clean_crop,cv2.COLOR_GRAY2BGR),overlay,truth_patch]
    scale=3
    panels=[cv2.resize(p,(p.shape[1]*scale,p.shape[0]*scale),interpolation=cv2.INTER_NEAREST) for p in panels]
    header=55; panel_w=panels[0].shape[1]; panel_h=panels[0].shape[0]
    sheet=np.full((panel_h+header,panel_w*4,3),255,np.uint8)
    titles=['observed','clean','observed + clean contour','derived truth']
    for i,(p,title) in enumerate(zip(panels,titles)):
        sheet[header:,i*panel_w:(i+1)*panel_w]=p
        cv2.putText(sheet,title,(i*panel_w+5,20),cv2.FONT_HERSHEY_SIMPLEX,.45,(20,20,20),1,cv2.LINE_AA)
    caption=f'{page} ({x},{y}) p={float(r.p_print):.4f} thin_proxy={bool(r.thin_continuous_proxy)}'
    cv2.putText(sheet,caption,(5,45),cv2.FONT_HERSHEY_SIMPLEX,.45,(20,20,20),1,cv2.LINE_AA)
    name=f'{page}_y{y}_x{x}_distance2_review.png'
    cv2.imwrite(str(CROPS/name),sheet)
    crop_records.append({'page':page,'x':x,'y':y,'file':name})

pd.DataFrame(crop_records).to_csv(AUDIT/'review_crop_inventory.csv',index=False)

# Contact sheets in manageable batches.
files=[CROPS/r['file'] for r in crop_records]
for batch_start in range(0,len(files),CONTACT_SHEET_ROWS):
    batch=files[batch_start:batch_start+CONTACT_SHEET_ROWS]
    if not batch: continue
    images=[cv2.imread(str(p),cv2.IMREAD_COLOR) for p in batch]
    width=max(im.shape[1] for im in images); height=sum(im.shape[0] for im in images)
    canvas=np.full((height,width,3),255,np.uint8); y0=0
    for im in images:
        canvas[y0:y0+im.shape[0],:im.shape[1]]=im; y0+=im.shape[0]
    cv2.imwrite(str(VISUAL/f'distance2_review_sheet_{batch_start//CONTACT_SHEET_ROWS+1:02d}.png'),canvas)

print('Review queue:',len(review),'Crops:',len(crop_records))

## 11. Final audit identity and summary

In [ ]:
environment={
    'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,
    'pandas':pd.__version__,'platform':platform.platform()
}
(OUT/'software_environment.json').write_text(json.dumps(environment,indent=2))
identity={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'version':'8.2c-whole-population-label-sensitivity-audit',
    'development_only':True,'promotion_authority':False,
    'retrained':False,'refitted':False,'recalibrated':False,'labels_changed':False,
    'source_v82_run':str(SOURCE),'source_record':source_record,
    'truth_replay':{
        'adaptive_block_size':ADAPTIVE_BLOCK_SIZE,'adaptive_c':ADAPTIVE_C,
        'truth_kernel_size':TRUTH_KERNEL_SIZE,'stored_truth_exactly_replayed':True,
    },
    'morphology_proxy_controls':answer['proxy_definition'],
    'primary_answer':answer,
    'enriched_table_sha256':sha256_file(TABLES/'all_eligible_label_sensitivity.parquet'),
    'annotation_distance2_table_sha256':sha256_file(TABLES/'annotation_distance2_candidates.parquet'),
    'software':environment,
}
(OUT/'experiment_identity.json').write_text(json.dumps(identity,indent=2,default=str))

lines=[
    'V8.2c WHOLE-POPULATION LABEL-SENSITIVITY AUDIT','',
    f'Source V8.2 run: {SOURCE}',f'Audit run: {OUT}',
    'Retrained: False','Refitted: False','Recalibrated: False','Labels changed: False','',
    f"Eligible current-ANNOTATION pixels: {answer['eligible_annotation_pixels']}",
    f"ANNOTATION pixels exactly two Chebyshev pixels from clean ink: {answer['annotation_exactly_two_chebyshev_pixels_from_clean_ink']}",
    f"Fraction: {answer['fraction_annotation_exactly_two_chebyshev_pixels_from_clean_ink']:.8f}",
    f"Distance-two ANNOTATION pixels meeting thin-continuous morphology proxy: {answer['distance2_annotation_thin_continuous_proxy_pixels']}",
    f"Proxy fraction within distance-two ANNOTATION: {answer['fraction_distance2_annotation_meeting_thin_continuous_proxy']:.8f}",'',
    'The morphology proxy is not a semantic staff-line/hairpin/stem/slur classifier.',
    'Alternative truth columns are sensitivity analyses only and were not used to retrain or rescore the CNN.',
]
(OUT/'executive_summary.txt').write_text('\n'.join(lines))
print('\n'.join(lines))
print('\nCOMPLETE:',OUT)

## Outputs to bring back

Start with:

- `executive_summary.txt`
- `primary_answer.json`
- `experiment_identity.json`
- `tables/truth_by_nearest_clean_ink_distance.csv`
- `tables/distance2_annotation_by_page.csv`
- `tables/distance2_annotation_by_source_group.csv`
- `tables/annotation_distance2_thin_continuous_proxy.csv`
- `tables/alternative_truth_transition_summary.csv`
- `tables/p_print_by_truth_and_clean_ink_distance.csv`
- `tables/distance2_annotation_review_queue.csv`
- `visual_audit/truth_distance_distribution.png`
- `visual_audit/annotation_distance2_fraction_by_page.png`
- the first two `visual_audit/distance2_review_sheet_*.png` files.